In [16]:
# ==============================================================================
# ETAPA 1: Inicialização e Leitura do Arquivo Real do Projeto
# ==============================================================================
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, month, year, to_date
from pyspark.sql.types import DoubleType
from pyspark.ml.feature import StringIndexer, VectorAssembler, Normalizer
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

spark = SparkSession.builder \
    .appName("PreparacaoModelagemVideos") \
    .getOrCreate()

# Lendo o arquivo Parquet que realmente existe na sua pasta trata_dev
df_raw = spark.read.parquet("videos-comments-tratados.snappy.parquet")

print("--- Schema Original Carregado ---")
df_raw.printSchema()
print(f"Total de linhas brutas (com comentários): {df_raw.count()}")

--- Schema Original Carregado ---
root
 |-- Video ID: string (nullable = true)
 |-- Title: string (nullable = true)
 |-- Published At: date (nullable = true)
 |-- Keyword: string (nullable = true)
 |-- Likes: integer (nullable = true)
 |-- Comments: integer (nullable = true)
 |-- Views: integer (nullable = true)
 |-- Interaction: integer (nullable = true)
 |-- Year: string (nullable = true)
 |-- Comment: string (nullable = true)
 |-- Sentiment: integer (nullable = true)
 |-- Likes Comment: integer (nullable = true)

Total de linhas brutas (com comentários): 18409


In [ ]:
# ==============================================================================
# ETAPA 2: Deduplicação por Vídeo e Seleção das Colunas Reais
# ==============================================================================
# Como o arquivo veio do join de vídeos x comentários, precisamos voltar ao nível
# de 1 linha por vídeo para não haver vazamento entre treino e teste.
colunas_video = ["Video ID", "Published At", "Keyword", "Likes", "Views", "Comments"]

df_video = df_raw.select(*colunas_video).dropDuplicates(["Video ID"])

print(f"Total de vídeos únicos após deduplicação: {df_video.count()}")

Total de vídeos únicos após deduplicação: 1869


In [18]:
# Leitura do diretório Parquet existente
df_video = spark.read.parquet("videos-tratados-parquet")

print("--- Schema Original ---")
df_video.printSchema()

print("--- Primeiros registros ---")
df_video.show(5, truncate=False)

print("Total de registros:", df_video.count())


Py4JJavaError: An error occurred while calling o451.parquet.
: java.lang.UnsatisfiedLinkError: 'boolean org.apache.hadoop.io.nativeio.NativeIO$Windows.access0(java.lang.String, int)'
	at org.apache.hadoop.io.nativeio.NativeIO$Windows.access0(Native Method)
	at org.apache.hadoop.io.nativeio.NativeIO$Windows.access(NativeIO.java:817)
	at org.apache.hadoop.fs.FileUtil.canRead(FileUtil.java:1415)
	at org.apache.hadoop.fs.FileUtil.list(FileUtil.java:1620)
	at org.apache.hadoop.fs.RawLocalFileSystem.listStatus(RawLocalFileSystem.java:802)
	at org.apache.hadoop.fs.FileSystem.listStatus(FileSystem.java:2079)
	at org.apache.hadoop.fs.FileSystem.listStatus(FileSystem.java:2123)
	at org.apache.hadoop.fs.ChecksumFileSystem.listStatus(ChecksumFileSystem.java:1020)
	at org.apache.spark.util.HadoopFSUtils$.listLeafFiles(HadoopFSUtils.scala:247)
	at org.apache.spark.util.HadoopFSUtils$.$anonfun$parallelListLeafFilesInternal$1(HadoopFSUtils.scala:159)
	at scala.collection.immutable.List.map(

In [19]:
# ==============================================================================
# ETAPA 3: Validação de Data, Tipagem Explícita e Limpeza de Nulos
# ==============================================================================
# Garante que 'Published At' seja reconhecido como data e extrai Mês e Ano
df_video = df_video.withColumn("Published At", to_date(col("Published At"))) \
                   .withColumn("Month", month(col("Published At")).cast(DoubleType())) \
                   .withColumn("Year", year(col("Published At")).cast(DoubleType()))

# Conversão explícita das métricas para DoubleType (exigência do Spark MLlib)
df_video = df_video.withColumn("Likes", col("Likes").cast(DoubleType())) \
                   .withColumn("Views", col("Views").cast(DoubleType())) \
                   .withColumn("Comments", col("Comments").cast(DoubleType()))

# Remove apenas registros onde os dados reais sejam nulos (sem inventar 0 para Comments)
df_video = df_video.na.drop(subset=["Likes", "Views", "Year", "Month", "Keyword", "Comments"])

print(f"Total de vídeos válidos prontos para divisão: {df_video.count()}")

Total de vídeos válidos prontos para divisão: 1869


In [ ]:
# ==============================================================================
# ETAPA 4: Divisão Treino (80%) e Teste (20%) ANTES de qualquer transformação ML
# ==============================================================================
train_data, test_data = df_video.randomSplit([0.8, 0.2], seed=42)

print(f"Registros de Treino: {train_data.count()}")
print(f"Registros de Teste: {test_data.count()}")

In [ ]:
# ==============================================================================
# ETAPA 5: Indexação Categórica ('Keyword') Ajustada SOMENTE no Treino
# ==============================================================================
# Atenção ao nome exato da coluna no seu schema: 'Keyword' com K maiúsculo
indexer = StringIndexer(
    inputCol="Keyword",
    outputCol="Keyword Index",
    handleInvalid="skip"
)

# Fit exclusivo no treino para impedir Data Leakage
indexer_model = indexer.fit(train_data)

train_data = indexer_model.transform(train_data)
test_data = indexer_model.transform(test_data)

In [ ]:
# ==============================================================================
# ETAPA 6: Montagem do Vetor de Features e Normalização (Sem PCA)
# ==============================================================================
assembler = VectorAssembler(
    inputCols=["Likes", "Views", "Year", "Month", "Keyword Index"],
    outputCol="Features",
    handleInvalid="error"
)

train_data = assembler.transform(train_data)
test_data = assembler.transform(test_data)

# Normalização L2 sobre as 5 features originais
normalizer = Normalizer(inputCol="Features", outputCol="Features Normal", p=2.0)

train_data = normalizer.transform(train_data)
test_data = normalizer.transform(test_data)

In [ ]:
# ==============================================================================
# ETAPA 7: Treinamento da Regressão Linear e Avaliação no Conjunto de Teste
# ==============================================================================
lr = LinearRegression(
    featuresCol="Features Normal",
    labelCol="Comments"
)

lr_model = lr.fit(train_data)
predictions = lr_model.transform(test_data)

# Avaliação oficial com RegressionEvaluator
evaluator_rmse = RegressionEvaluator(labelCol="Comments", predictionCol="prediction", metricName="rmse")
evaluator_r2 = RegressionEvaluator(labelCol="Comments", predictionCol="prediction", metricName="r2")

print("--- Métricas Baseline (Regressão Linear sem PCA) ---")
print(f"RMSE: {evaluator_rmse.evaluate(predictions):.4f}")
print(f"R²:   {evaluator_r2.evaluate(predictions):.4f}")

predictions.select("Video ID", "Keyword", "Likes", "Views", "Comments", "prediction").show(10)

In [ ]:
# ==============================================================================
# ETAPA 8: Gravação Segura em Parquet no Windows (Contornando erro HADOOP_HOME)
# ==============================================================================
# Como o Spark nativo no Windows exige winutils.exe (Hadoop) para gravar Parquet,
# convertemos as colunas tabulares/previsões para Pandas e salvamos via PyArrow/Snappy.
# (Vetores internos do Spark ML são convertidos para string ou removidos antes do export)

df_export = predictions.drop("Features", "Features Normal").toPandas()
df_export.to_parquet("videos-preparados.snappy.parquet", index=False)

print("Arquivo 'videos-preparados.snappy.parquet' salvo com sucesso no Windows!")